In [1]:
import json
import logging
import os
import time
from datetime import datetime
from pathlib import Path

import config
from llm import LLMOrchestrator
from agents.state import OrchestratorState, DiagnosticianReport
from agents import db_search, selector as selector_agent, diagnostician as diag_agent
from mechanism_builder import build_mechanism_yaml, dispatch as builder_dispatch
from tools import cantera_tool


In [2]:
def _print_header(state: OrchestratorState, llm_selector: LLMOrchestrator, llm_diagnostician: LLMOrchestrator):
    print(f"\n{'='*60}")
    print(f"  CombustionAgent — Multi-Agent Mode")
    print(f"  Fuel: {state.fuel} | Run: {state.run_id}")
    print(f"  Selector:     {llm_selector.backend}/{llm_selector.model}")
    print(f"  Diagnostician: {llm_diagnostician.backend}/{llm_diagnostician.model}")
    print(f"  Max iters: {state.max_iters} | Threshold: {state.acceptance_threshold:.0%}")
    print(f"{'='*60}")
    
def _save_intermediate(state: OrchestratorState, yaml: str, iteration: int):
        run_dir = state.run_id
        run_dir.mkdir(exist_ok=True)
        path = run_dir / f"mechanism_iter{iteration:02d}.yaml"
        path.write_text(yaml)
        
def _is_third_body(rxn: dict) -> bool:
    """Use kinetics_type as authoritative signal; fall back to label string."""
    kt = rxn.get("kinetics_type", "")
    if kt in ("ThirdBody", "Troe", "Lindemann", "PDepArrhenius", "MultiPDepArrhenius"):
        return True
    label_up = rxn.get("label", "").upper()
    return "(+M)" in label_up or "+ M " in label_up or label_up.strip().endswith("+M")

In [3]:
fuel = "CH4"
max_iters = 10
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

llm_selector = LLMOrchestrator(backend="openrouter")
llm_diagnostician = LLMOrchestrator(backend="openrouter")


In [4]:
state = OrchestratorState(
    fuel=fuel,
    run_id=run_id,
    max_iters=max_iters,
)



In [5]:
_print_header(state, llm_selector, llm_diagnostician)


  CombustionAgent — Multi-Agent Mode
  Fuel: CH4 | Run: 20260406_203816
  Selector:     openrouter/anthropic/claude-opus-4-5
  Diagnostician: openrouter/anthropic/claude-opus-4-5
  Max iters: 10 | Threshold: 75%


In [6]:
it = state.iteration
print(f"\n{'─'*60}")
print(f"  ITERATION {it} | current={len(state.current_reactions)} rxns | best={state.best_score:.1%}")
print(f"{'─'*60}")


────────────────────────────────────────────────────────────
  ITERATION 0 | current=0 rxns | best=0.0%
────────────────────────────────────────────────────────────


In [7]:
context = state.context_for_db_search()

In [8]:
from tools import db_retrieval

In [9]:
context = state.context_for_db_search()

target_species    = context["target_species"]
already_tried     = set(context["already_tried_labels"])
current_labels    = set(context["current_reaction_labels"])
library_pref      = context["library_preference"]

In [10]:
result_str = db_retrieval.dispatch("get_reactions_for_fuel", {
        "fuel_species":        context["target_species"],
        "library_preference":  context["library_preference"],
        "limit":               300,
    })
result = json.loads(result_str)
all_reactions = result.get("reactions", [])

In [11]:
from mechanism_builder import NASA7_DATA

# Species with known thermo data — only reactions whose every species appears
# in this set can be loaded by Cantera without error.
_KNOWN_SPECIES = set(NASA7_DATA.keys())

print(_KNOWN_SPECIES)

{'C2H', 'OH', 'CO2', 'AR', 'CH3', 'CH2O', 'HE', 'CH2(S)', 'O2', 'C2H6', 'H2O2', 'CO', 'CH', 'H', 'C2H2', 'C2H4', 'CH2', 'H2', 'HO2', 'H2O', 'N2', 'C2H3', 'O', 'C2H5', 'HCO', 'CH4'}


In [12]:
# Filter: remove already-tried and already-in-mechanism
excluded = already_tried | current_labels
candidates = [
    r for r in all_reactions
    if r["label"] not in excluded
]

# Filter to reactions whose every species has thermo data — prevents the
# LLM from selecting CO/C2H4/etc. reactions that can never load in Cantera.
def _all_species_known(rxn: dict) -> bool:
    species = {sp.upper() for sp in rxn.get("reactants", []) + rxn.get("products", [])}
    return species.issubset(_KNOWN_SPECIES)

before = len(candidates)
candidates = [r for r in candidates if _all_species_known(r)]



In [14]:
print(len(all_reactions))

300


In [15]:
# ── Step 1: DB Search ────────────────────────────────────────────────
print(f"\n[1/5] DB Search → refreshing candidate pool")
# On refinement iterations, add diagnostician targets to search
if state.diagnostician_report:
    print(state.diagnostician_report)
    extra = state.diagnostician_report.target_species
    for sp in extra:
        if sp not in state.target_species:
            state.target_species.append(sp)

candidates = db_search.run(state)


[1/5] DB Search → refreshing candidate pool


In [16]:
print(len(candidates))

300


In [17]:
candidates

[{'label': 'H + O2 <=> O + OH',
  'reactants': ['H', 'O2'],
  'products': ['O', 'OH'],
  'kinetics': "Arrhenius(A=[2.644e+16, 'cm^3/(mol*s)'], n=-0.6707, Ea=[17041, 'cal/mol'], T0=[1, 'K'])",
  'kinetics_type': 'Arrhenius',
  'library': 'JetSurF2.0',
  'rank': None,
  'duplicate': None},
 {'label': 'O + H2 <=> H + OH',
  'reactants': ['O', 'H2'],
  'products': ['H', 'OH'],
  'kinetics': "Arrhenius(A=[45890, 'cm^3/(mol*s)'], n=2.7, Ea=[6260, 'cal/mol'], T0=[1, 'K'])",
  'kinetics_type': 'Arrhenius',
  'library': 'JetSurF2.0',
  'rank': None,
  'duplicate': None},
 {'label': 'OH + H2 <=> H + H2O',
  'reactants': ['OH', 'H2'],
  'products': ['H', 'H2O'],
  'kinetics': "Arrhenius(A=[173400000.0, 'cm^3/(mol*s)'], n=1.51, Ea=[3430, 'cal/mol'], T0=[1, 'K'])",
  'kinetics_type': 'Arrhenius',
  'library': 'JetSurF2.0',
  'rank': None,
  'duplicate': None},
 {'label': 'OH + OH <=> O + H2O',
  'reactants': ['OH', 'OH'],
  'products': ['O', 'H2O'],
  'kinetics': "Arrhenius(A=[39730, 'cm^3/(mol*s)'

In [ ]:
third_bodies = []
for rxn in candidates:
    if _is_third_body(rxn):
        third_bodies.append(rxn)

print(f"      Pool: {len(candidates)} candidates "
              f"({sum(1 for r in candidates if _is_third_body(r))} third-body)")


In [ ]:
third_bodies

In [ ]:

state.candidate_pool = candidates

In [ ]:
state.candidate_pool

In [ ]:
# ── Step 2: Chemistry Selector ───────────────────────────────────────
print(f"\n[2/5] Chemistry Selector → choosing reactions to add")
sel_output = selector_agent.run(state, llm_selector)

In [ ]:
sel_output

In [ ]:
selected_labels = [r["label"] for r in sel_output.selected]
print(f"      Selected: {selected_labels}")
print(f"      Reason:   {sel_output.reasoning[:200]}")

In [ ]:
state.add_reactions(sel_output.selected, reason=sel_output.reasoning[:300])

In [ ]:
if sel_output.next_target_species:
    for sp in sel_output.next_target_species:
        if sp not in state.target_species:
            print(f"      Adding target species: {sp}")
            state.target_species.append(sp)

In [ ]:
print(f"\n[3/5] Mechanism Builder → assembling YAML")
result_str = builder_dispatch("build_mechanism", {
    "reactions":      state.current_reactions,
    "mechanism_name": f"CombustionAgent {state.fuel} iter={it}",
    "description":    f"Multi-agent iteration {it}",
    "extra_species":  ["N2", "AR"],
})
build_result = json.loads(result_str)

In [ ]:
yaml = build_result['mechanism_yaml']



In [ ]:
print(yaml)

In [ ]:
val_result = cantera_tool.validate_mechanism(
            yaml,
            conditions=state.target_conditions,
            compute_lfs=False,
        )